# Fine-tune Laguna XS 2.1 on Zypher data (Google Colab)

Base model: [poolside/Laguna-XS-2.1](https://huggingface.co/poolside/Laguna-XS-2.1) — 33B MoE coding model (3B active/token).

**Runtime:** Change runtime type → **A100** (recommended) or **L4** (24 GB+).

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {mem:.1f} GB')
    if mem < 20:
        print('WARNING: Laguna QLoRA needs ~24 GB+. Use A100 or L4 runtime.')
else:
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → GPU')

In [ ]:
REPO = 'https://github.com/FOUNDEROF-AIRIES-AGENT/Zypher-Training-data.git'
BRANCH = 'cursor/llm-finetune-ready-da1e'

!rm -rf Zypher-Training-data
!git clone --branch {BRANCH} --single-branch {REPO}
%cd Zypher-Training-data

In [ ]:
!pip install -q -r requirements.txt
# Laguna needs recent transformers
!pip install -q -U transformers accelerate peft trl bitsandbytes

In [ ]:
# Hugging Face login (required if model is gated — accept license on HF first)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!make generate-smoke
!make prepare-advanced
!python3 scripts/validate_pipeline.py

## Fine-tune Laguna XS 2.1 with QLoRA

Uses `config/training_laguna.yaml`. First run downloads ~66 GB (4-bit loads subset into VRAM).

In [ ]:
!python3 scripts/train.py --config config/training_laguna.yaml

In [ ]:
!python3 scripts/infer.py \
  --base-model poolside/Laguna-XS-2.1 \
  --adapter outputs/laguna-xs-2.1-sft/final \
  --question "Explain GraphRAG for code architecture."

In [ ]:
!zip -r laguna-zyphere-adapter.zip outputs/laguna-xs-2.1-sft/final
from google.colab import files
files.download('laguna-zyphere-adapter.zip')